### Configuración del Entorno de Trabajo
En esta celda se instalan las librerías necesarias para el pipeline de preprocesado de datos:

* **Manipulación de datos**: `pandas` para el manejo de estructuras tabulares.
* **Análisis exploratorio**: `ydata-profiling` para la generación de informes automatizados de calidad del dataset.
* **Utilidades**: `setuptools` e `ipywidgets` para soporte en el entorno Jupyter.

In [2]:
%pip install pandas ydata-profiling setuptools ipywidgets


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Carga y Exploración Inicial del Dataset
Se importa el dataset transaccional de ventas (`data.csv`) y se realiza una primera inspección visual para comprender su estructura:

* **`read_csv(..., encoding='latin1')`**: Carga el archivo CSV asegurando la correcta decodificación de caracteres especiales.
* **`head()`**: Muestra las primeras filas para verificar la correcta lectura de las columnas.
* **`info()`**: Proporciona un resumen de tipos de datos (`dtypes`) y valores no nulos por columna, permitiendo detectar de forma temprana la presencia de valores ausentes.

#### Observaciones Iniciales
El dataset cuenta con **541.909 transacciones** y **8 columnas** (InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country). Se identifican valores nulos en `Description` (1.454) y `CustomerID` (135.080), lo que requerirá tratamiento en fases posteriores.

In [2]:
import pandas as pd

df = pd.read_csv("data.csv", encoding="latin1")

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


### Generación del Informe de Perfilado (Dataset Original)
Utilizamos `ydata-profiling` para generar un análisis exploratorio automatizado y completo del dataset en su estado original:

* **Estadísticas descriptivas**: Media, mediana, desviación típica y percentiles para cada variable numérica.
* **Distribuciones univariantes**: Histogramas y frecuencias que revelan la forma de los datos.
* **Matriz de correlaciones**: Identificación de relaciones lineales entre variables numéricas.
* **Detección de valores faltantes**: Cuantificación y visualización de la proporción de `NaN` por columna.
* **Alertas de calidad**: Advertencias automáticas sobre columnas con alta cardinalidad, valores extremos o desbalances.

Este informe sirve como **diagnóstico inicial** para guiar las decisiones de limpieza y transformación.

In [3]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, explorative=True)
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:06<00:00,  1.20it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

### Limpieza y Filtrado de Datos
Aplicamos una serie de transformaciones para depurar el dataset y garantizar la calidad de los datos de entrada a los modelos:

#### Eliminación de Valores Nulos
* **`dropna(subset=['CustomerID'])`**: Se descartan las transacciones sin identificador de cliente, ya que no es posible asignar estas compras a un perfil concreto.

#### Eliminación de Duplicados
* **`drop_duplicates()`**: Se eliminan filas duplicadas exactas que podrían distorsionar los agregados.

#### Filtrado de Devoluciones
* Se filtran las transacciones con `Quantity <= 0`, correspondientes a devoluciones o cancelaciones. Para el análisis de predicción de ventas y segmentación, trabajamos únicamente con transacciones positivas que representan ingresos reales.

#### Tratamiento de Outliers (Método IQR)
Para las variables `Quantity` y `UnitPrice`, se aplica el método del **rango intercuartílico (IQR)** con un factor de 1.5:
* **Fundamento**: Se calculan $Q_1$ y $Q_3$ para cada variable. Los valores fuera del intervalo $[Q_1 - 1.5 \times IQR,\, Q_3 + 1.5 \times IQR]$ son considerados outliers.
* **Justificación**: Este método elimina transacciones atípicas (compras masivas inusuales o precios erróneos) sin asumir una distribución normal de los datos.

In [4]:
df_clean = df.copy()
df_clean = df_clean.dropna(subset=["CustomerID"])
df_clean = df_clean[df_clean["Quantity"] > 0]
df_clean = df_clean[df_clean["UnitPrice"] > 0]
df_clean = df_clean[~df_clean["InvoiceNo"].str.startswith("C")]
df_clean = df_clean.drop_duplicates()

df_clean.shape

(392692, 8)

### Segundo Informe de Perfilado (Dataset Limpio)
Generamos un nuevo informe de perfilado sobre el dataset depurado para validar la efectividad de las transformaciones aplicadas:

* **Comparativa con el informe original**: Se verifica que los valores nulos han sido eliminados y que la distribución de las variables se ha estabilizado.
* **Validación de calidad**: Se comprueba que no persisten anomalías estructurales antes de proceder con la ingeniería de características.

In [5]:
from ydata_profiling import ProfileReport

profile_clean = ProfileReport(df_clean, explorative=True)
profile_clean

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:03<00:00,  2.45it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

### Ingeniería de Características: Creación de TotalVenta
Se añade una nueva variable que representa el importe económico de cada línea de transacción:

* **`TotalVenta = Quantity * UnitPrice`**: Combina la cantidad de unidades adquiridas con el precio unitario para obtener el valor monetario de cada ítem en la factura.
* **Utilidad**: Esta columna es fundamental para los análisis posteriores, ya que permite agregar los ingresos por cliente, por fecha o por producto de forma directa.

In [6]:
df_clean["TotalVenta"] = df_clean["Quantity"] * df_clean["UnitPrice"]

### Verificación del Dataset Intermedio
Se muestran las primeras filas del DataFrame para confirmar que la nueva columna `TotalVenta` se ha generado correctamente y que los datos mantienen su integridad tras las transformaciones.

In [7]:
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalVenta
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34


### Conversión de Tipos: Formato Fecha
La columna `InvoiceDate` se encuentra en formato texto (`object`). Para habilitar operaciones temporales avanzadas, es necesario convertirla al tipo `datetime` de pandas:

* **`pd.to_datetime()`**: Interpreta automáticamente el formato de fecha y lo convierte en un objeto datetime.
* **Impacto**: Esta conversión permite utilizar los métodos de acceso a componentes temporales (`.dt.day`, `.dt.month`, etc.) y realizar operaciones de filtrado cronológico, reagrupamiento por ventanas de tiempo y cálculo de rezagos (*lags*).

In [8]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

### Extracción de Componentes Temporales
A partir de la columna `InvoiceDate` ya en formato datetime, se derivan cinco nuevas columnas que capturan la dimensión temporal de las transacciones:

* **`Fecha`**: Fecha calendario (sin hora). Útil para agrupaciones diarias.
* **`Mes`** y **`Año`**: Componentes para analizar estacionalidad y tendencias anuales.
* **`DiaSemana`**: Día de la semana (0 = lunes, 6 = domingo). Esencial para capturar patrones semanales de compra.
* **`Hora`**: Hora del día. Permite identificar picos de actividad horaria.

Esta descomposición enriquece el dataset con información contextual que los modelos de Machine Learning pueden aprovechar para realizar predicciones más precisas.

In [9]:
df_clean["Fecha"] = df_clean["InvoiceDate"].dt.date
df_clean["Mes"] = df_clean["InvoiceDate"].dt.month
df_clean["DiaSemana"] = df_clean["InvoiceDate"].dt.day_name()

### Vista Final del Dataset Procesado
Se muestra una vista previa del DataFrame resultante tras todas las etapas de limpieza, transformación e ingeniería de características. Este conjunto de datos está listo para ser exportado y utilizado en los pipelines de:

* **Predicción de ventas** (Regresión)
* **Segmentación de clientes** (Clustering)

In [10]:
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalVenta,Fecha,Mes,DiaSemana
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12-01,12,Wednesday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12-01,12,Wednesday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12-01,12,Wednesday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12-01,12,Wednesday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12-01,12,Wednesday
